In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


spark = SparkSession.builder \
    .appName("Spark SCD2 Pipeline") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .config("spark.sql.shuffle.partitions", "6") \
    .config("spark.streaming.kafka.maxRatePerPartition", "10000") \
    .getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", "4")

spark



In [2]:
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker2:29094") \
    .option("subscribe", "bronze.mysql") \
    .option("startingOffsets", "earliest") \
    .load()

In [3]:
cdc_schema = StructType([
    StructField("before", StructType([
        StructField("customer_id", IntegerType()),
        StructField("full_name", StringType()),
        StructField("email", StringType()),
        StructField("status", StringType())
    ])),
    StructField("after", StructType([
        StructField("customer_id", IntegerType()),
        StructField("full_name", StringType()),
        StructField("email", StringType()),
        StructField("status", StringType())
    ])),
    StructField("op", StringType()),
    StructField("ts_ms", StringType())
])

In [5]:
parsed_df = kafka_df.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), cdc_schema).alias("data")) \
    .select("data.before", "data.after", "data.op", "data.ts_ms")

In [6]:
silver_df = parsed_df.filter(col("op").isin("c", "u")) \
    .select(
        col("after.customer_id").alias("customer_id"),
        col("after.full_name").alias("full_name"),
        col("after.email").alias("email"),
        col("after.status").alias("status"),
        current_timestamp().alias("start_date"),
        lit(None).cast(TimestampType()).alias("end_date"),
        lit(True).alias("is_current")
    )

# ----------------------------
# 6. Write to PostgreSQL
# ----------------------------
postgres_url = "jdbc:postgresql://postgresDB:5432/ECOMMERCE"
postgres_properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

query = silver_df.writeStream \
    .foreachBatch(lambda df, epoch_id: df.write \
        .jdbc(url=postgres_url,
              table="customers_history",
              mode="append",
              properties=postgres_properties)) \
    .outputMode("update") \
    .start()

query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = 131c28bc-1534-453a-86d1-df56aa61d414, runId = 958d4572-028f-4a08-9800-852b4c363d2b] terminated with exception: org/apache/spark/kafka010/KafkaConfigUpdater